In [62]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from statsmodels.tools.eval_measures import rmse
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb
import catboost
import mlflow

mlflow.autolog()

def mape(y_true, y_pred, *, ignore_zeros=True):
    """
    Calculate Mean Absolute Percentage Error (MAPE).

    Parameters
    ----------
    y_true : array-like
        True values.
    y_pred : array-like
        Predicted values.
    ignore_zeros : bool, default True
        If True, excludes observations where y_true == 0.
        If False, raises an error when y_true contains zeros.

    Returns
    -------
    float
        MAPE value in percentage.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if y_true.shape != y_pred.shape:
        raise ValueError("y_true and y_pred must have the same shape")

    if ignore_zeros:
        mask = y_true != 0
        if not np.any(mask):
            raise ValueError("All y_true values are zero; MAPE is undefined")
        y_true = y_true[mask]
        y_pred = y_pred[mask]
    else:
        if np.any(y_true == 0):
            raise ValueError("y_true contains zeros; MAPE is undefined")

    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

2026/03/08 19:08:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.
2026/03/08 19:08:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/03/08 19:08:47 WARNING mlflow.utils.autologging_utils: MLflow statsmodels autologging is known to be compatible with 0.14.1 <= statsmodels <= 0.14.4, but the installed version is 0.14.6. If you encounter errors during autologging, try upgrading / downgrading statsmodels to a compatible version, or try upgrading MLflow.
2026/03/08 19:08:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2026/03/08 19:08:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for xgboost.


In [63]:
df1_dict = pd.read_excel('data/raw/full2024.xlsx', sheet_name='Dict')
df2_dict = pd.read_excel('data/raw/8month2025.xlsx', sheet_name='Dict')
dict_df = pd.concat([df1_dict, df2_dict])
dict_df.drop_duplicates(subset=['EIC-код'], inplace=True)
dict_df.reset_index(drop=True, inplace=True)
dict_df

,EIC-код,Унікод,АЗС,Тип,Область,Адреса,GPS-координати - Широта,GPS-координати - Довгота,ОСР код,ОСР опис
0,62Z3386096691495,40110200,АЗС_02,ОККО-міська,Львівська,"м. Львів, вул. Замарстинівська, 174",49.871014,24.020029,MGA-00900,Львів
1,62Z5700239654314,40110500,АЗС_05,ОККО-міська,Львівська,"м. Львів, вул. Клепарівська, 30 Б",49.853477,24.014784,MGA-00900,Львів
2,62Z8292498523912,40110700,АЗС_07,ОККО-міська,Львівська,"м. Львів, вул. Володимира Великого, 58",49.812439,23.984974,MGA-00900,Львів
3,62Z6337618908569,40111000,АЗС_10,ОККО-стандарт,Львівська,"м. Львів, вул. Липинського, 54 А",49.864338,24.039696,MGA-00900,Львів
4,62Z1180469265339,40111100,АЗС_11,ОККО-міська,Львівська,"Львів, вул. Джорджа Вашингтона, 12",49.813316,24.077612,MGA-00900,Львів
...,...,...,...,...,...,...,...,...,...,...
424,62Z2673921373211,41261600,АЗС_39,ОККО-трасова,Дніпропетровська,"с/р Новоолександрівська, вздовж АД М-04 Знам'я...",48.430910,34.944208,MGA-02400,Дніпро
425,62Z7841956124522,41062200,АЗС_36,ОККО-стандарт,Дніпропетровська,"м. Дніпро, вул. Каштанова, 4 К",48.501564,35.080823,MGA-02400,Дніпро
426,62Z5821574095639,41261800,АЗС_45,ОККО-міська,Дніпропетровська,"м. Дніпро, Полтавське шосе, 11 К",48.519610,34.989180,MGA-02400,Дніпро
427,62Z1528625762710,41261400,АЗС_46,ОККО-міська,Дніпропетровська,"м. Дніпро, вул. Робоча, 84",48.454806,35.003711,MGA-02400,Дніпро


In [64]:
df = pd.read_parquet('data/raw/full2024_8month2025.parquet')

df.dropna(inplace=True)

cols_to_int = ['Year', 'Month', 'Day', 'Hour']
for col in cols_to_int:
    df[col] = df[col].astype(int)

df = df.merge(dict_df, on='EIC-код', how='left', suffixes=('', '_dict'))
df

,EIC-код,Група,Дата,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,__source_file,Унікод,АЗС,Тип,Область,Адреса,GPS-координати - Широта,GPS-координати - Довгота,ОСР код,ОСР опис
0,62Z0008583037334,Б,2024-12-01,2024,12,1,1,20.716983,2.15221,6.136750,12_2024.xlsx,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород
1,62Z0008583037334,Б,2024-12-01,2024,12,1,2,19.365875,2.15221,6.136750,12_2024.xlsx,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород
2,62Z0008583037334,Б,2024-12-01,2024,12,1,3,18.529475,2.15221,6.136750,12_2024.xlsx,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород
3,62Z0008583037334,Б,2024-12-01,2024,12,1,4,18.207783,2.15221,6.136750,12_2024.xlsx,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород
4,62Z0008583037334,Б,2024-12-01,2024,12,1,5,17.886091,2.15221,6.136750,12_2024.xlsx,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5454708,62Z9997819406173,Б,2024-01-31,2024,1,31,20,25.794128,1.63103,3.662765,1_2024.xlsx,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів
5454709,62Z9997819406173,Б,2024-01-31,2024,1,31,21,25.204357,1.63103,3.662765,1_2024.xlsx,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів
5454710,62Z9997819406173,Б,2024-01-31,2024,1,31,22,23.646472,1.63103,3.662765,1_2024.xlsx,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів
5454711,62Z9997819406173,Б,2024-01-31,2024,1,31,23,21.532199,1.63103,3.662765,1_2024.xlsx,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів


In [65]:
df['Дата'] = pd.to_datetime(df['Дата'], errors='coerce')
df['datetime'] = df['Дата'] + pd.to_timedelta(df['Hour'].fillna(0), unit='h')

In [66]:
val = df[(df['Дата'] > '2025-06-30') & (df['Дата'] < '2025-08-01')]
val

,EIC-код,Група,Дата,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,...,Унікод,АЗС,Тип,Область,Адреса,GPS-координати - Широта,GPS-координати - Довгота,ОСР код,ОСР опис,datetime
1579464,62Z0008583037334,А,2025-07-01,2025,7,1,1,19.000000,2.37385,5.441006,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2025-07-01 01:00:00
1579465,62Z0008583037334,А,2025-07-01,2025,7,1,2,16.000000,2.37385,5.441006,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2025-07-01 02:00:00
1579466,62Z0008583037334,А,2025-07-01,2025,7,1,3,16.000000,2.37385,5.441006,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2025-07-01 03:00:00
1579467,62Z0008583037334,А,2025-07-01,2025,7,1,4,15.000000,2.37385,5.441006,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2025-07-01 04:00:00
1579468,62Z0008583037334,А,2025-07-01,2025,7,1,5,14.000000,2.37385,5.441006,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2025-07-01 05:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2168707,62Z9997819406173,Б,2025-07-31,2025,7,31,20,23.504910,1.76339,5.441006,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2025-07-31 20:00:00
2168708,62Z9997819406173,Б,2025-07-31,2025,7,31,21,23.583325,1.76339,5.441006,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2025-07-31 21:00:00
2168709,62Z9997819406173,Б,2025-07-31,2025,7,31,22,24.475296,1.76339,5.441006,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2025-07-31 22:00:00
2168710,62Z9997819406173,Б,2025-07-31,2025,7,31,23,23.259863,1.76339,5.441006,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2025-07-31 23:00:00


In [67]:
test = df[df['Дата'] > '2025-07-31']
test

,EIC-код,Група,Дата,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,...,Унікод,АЗС,Тип,Область,Адреса,GPS-координати - Широта,GPS-координати - Довгота,ОСР код,ОСР опис,datetime
1580208,62Z0008583037334,А,2025-08-01,2025,8,1,1,18.000000,2.37385,5.447321,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2025-08-01 01:00:00
1580209,62Z0008583037334,А,2025-08-01,2025,8,1,2,16.000000,2.37385,5.447321,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2025-08-01 02:00:00
1580210,62Z0008583037334,А,2025-08-01,2025,8,1,3,14.000000,2.37385,5.447321,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2025-08-01 03:00:00
1580211,62Z0008583037334,А,2025-08-01,2025,8,1,4,15.000000,2.37385,5.447321,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2025-08-01 04:00:00
1580212,62Z0008583037334,А,2025-08-01,2025,8,1,5,14.000000,2.37385,5.447321,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2025-08-01 05:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2169451,62Z9997819406173,Б,2025-08-31,2025,8,31,20,22.540932,1.76339,5.447321,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2025-08-31 20:00:00
2169452,62Z9997819406173,Б,2025-08-31,2025,8,31,21,24.207382,1.76339,5.447321,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2025-08-31 21:00:00
2169453,62Z9997819406173,Б,2025-08-31,2025,8,31,22,25.211150,1.76339,5.447321,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2025-08-31 22:00:00
2169454,62Z9997819406173,Б,2025-08-31,2025,8,31,23,22.579913,1.76339,5.447321,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2025-08-31 23:00:00


In [68]:
train = df[df['Дата'] < '2025-07-01']
train

,EIC-код,Група,Дата,Year,Month,Day,Hour,Sum of кВт,Average of Ціна розподілу ЕЕ грн. без ПДВ/кВт*год,Average of Ціна ЕЕ грн. без ПДВ/кВт*год,...,Унікод,АЗС,Тип,Область,Адреса,GPS-координати - Широта,GPS-координати - Довгота,ОСР код,ОСР опис,datetime
0,62Z0008583037334,Б,2024-12-01,2024,12,1,1,20.716983,2.15221,6.136750,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2024-12-01 01:00:00
1,62Z0008583037334,Б,2024-12-01,2024,12,1,2,19.365875,2.15221,6.136750,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2024-12-01 02:00:00
2,62Z0008583037334,Б,2024-12-01,2024,12,1,3,18.529475,2.15221,6.136750,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2024-12-01 03:00:00
3,62Z0008583037334,Б,2024-12-01,2024,12,1,4,18.207783,2.15221,6.136750,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2024-12-01 04:00:00
4,62Z0008583037334,Б,2024-12-01,2024,12,1,5,17.886091,2.15221,6.136750,...,40312900,АЗС_29,ОККО-комплекс,Закарпатська,"Ужгородський р-н, с. Соломоново, автодорога Ки...",48.442430,22.192190,MGA-00700,Ужгород,2024-12-01 05:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5454708,62Z9997819406173,Б,2024-01-31,2024,1,31,20,25.794128,1.63103,3.662765,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2024-01-31 20:00:00
5454709,62Z9997819406173,Б,2024-01-31,2024,1,31,21,25.204357,1.63103,3.662765,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2024-01-31 21:00:00
5454710,62Z9997819406173,Б,2024-01-31,2024,1,31,22,23.646472,1.63103,3.662765,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2024-01-31 22:00:00
5454711,62Z9997819406173,Б,2024-01-31,2024,1,31,23,21.532199,1.63103,3.662765,...,40116500,АЗС_65,ОККО-міська,Львівська,"м. Львів, вул. Зелена, 283",49.800007,24.068964,MGA-00900,Львів,2024-01-31 23:00:00


In [69]:
print(f"{len(train)} + {len(val)} + {len(test)} = {len(train) + len(val) + len(test)}, and should be {len(df)}")
if len(train) + len(val) + len(test) == len(df):
    print("All correct!")

4864721 + 295368 + 294624 = 5454713, and should be 5454713
All correct!


In [70]:
y_col = "Sum of кВт"

features = ['Year', 'Month', 'Day', 'Hour', 'Унікод', 'GPS-координати - Широта', 'GPS-координати - Довгота']

In [71]:
X = train[features]
y = train[y_col]

In [72]:
with mlflow.start_run(run_name="linear_regression_baseline"):
    lr_model = LinearRegression()
    lr_model.fit(X, y)

    val_preds = lr_model.predict(val[features])

    val_rmse = rmse(val[y_col], val_preds)
    val_mape = mape(val[y_col], val_preds)

    print(f"RMSE: {val_rmse:.3f}")
    print(f"MAPE: {val_mape:.3f}")

    mlflow.log_metric("val_rmse", float(val_rmse))
    mlflow.log_metric("val_mape", float(val_mape))
    mlflow.log_param("features", ",".join(features))

2026/03/08 19:09:05 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\Lev\Miniconda3\envs\Diploma\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/03/08 19:09:05 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\Lev\Miniconda3\envs\Diploma\lib\si

RMSE: 17.268
MAPE: 153.952


In [73]:
# XGBoost
with mlflow.start_run(run_name="XGBoost_regression_baseline"):

    xgb_model = xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
    xgb_model.fit(X, y)

    val_preds = xgb_model.predict(val[features])

    val_rmse = rmse(val[y_col], val_preds)
    val_mape = mape(val[y_col], val_preds)

    print(f"RMSE: {val_rmse:.3f}")
    print(f"MAPE: {val_mape:.3f}")

    mlflow.log_metric("val_rmse", float(val_rmse))
    mlflow.log_metric("val_mape", float(val_mape))
    mlflow.log_param("features", ",".join(features))

2026/03/08 19:09:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\Lev\Miniconda3\envs\Diploma\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/03/08 19:09:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/08 19:09:27 WARNING mlflow.util

RMSE: 9.710
MAPE: 48.548


In [74]:
# LightGBM
with mlflow.start_run(run_name="LightGBM_regression_baseline"):

    lgb_model = lgb.LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1)
    lgb_model.fit(X, y)

    val_preds = lgb_model.predict(val[features])

    val_rmse = rmse(val[y_col], val_preds)
    val_mape = mape(val[y_col], val_preds)

    print(f"RMSE: {val_rmse:.3f}")
    print(f"MAPE: {val_mape:.3f}")

    mlflow.log_metric("val_rmse", float(val_rmse))
    mlflow.log_metric("val_mape", float(val_mape))
    mlflow.log_param("features", ",".join(features))

2026/03/08 19:09:40 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\Lev\Miniconda3\envs\Diploma\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/03/08 19:09:47 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "C:\Users\Lev\Miniconda3\envs\Diploma\lib\si

RMSE: 9.843
MAPE: 52.158


In [75]:
y_col = "Sum of кВт"

features = ['Year', 'Month', 'Day', 'Hour', 'Унікод', 'GPS-координати - Широта', 'GPS-координати - Довгота']
X_categorical = train
y = train[y_col]

# CatBoost
with mlflow.start_run(run_name="CatBoost_regression_baseline"):

    cat_model = catboost.CatBoostRegressor(iterations=200, depth=6, learning_rate=0.1, random_state=42, verbose=0)
    cat_model.fit(X, y)

    val_preds = cat_model.predict(val[features])

    val_rmse = rmse(val[y_col], val_preds)
    val_mape = mape(val[y_col], val_preds)

    print(f"RMSE: {val_rmse:.3f}")
    print(f"MAPE: {val_mape:.3f}")

    mlflow.log_metric("val_rmse", float(val_rmse))
    mlflow.log_metric("val_mape", float(val_mape))
    mlflow.log_param("features", ",".join(features))

RMSE: 10.937
MAPE: 68.397
